# Fooocus-API on Colab
Runtime > Change runtime type > GPU, then Run all. Copy the printed URL into `FOOOCUS_API_URL` in the app's `.env`.

**The URL has no login. Anyone who has it can use this GPU session, so keep it private.**

In [ ]:
!nvidia-smi -L || echo "No GPU: switch the runtime type to GPU"

In [ ]:
%cd /content
!test -d Fooocus-API || git clone https://github.com/mrhan1993/Fooocus-API
%cd /content/Fooocus-API
!pip install -q -r requirements.txt

In [ ]:
import subprocess, time, urllib.request
log = open("/content/fooocus-api.log", "w")
server = subprocess.Popen(["python", "main.py", "--host", "127.0.0.1", "--port", "8888"],
                          cwd="/content/Fooocus-API", stdout=log, stderr=subprocess.STDOUT)
# First start downloads ~10 GB of models; allow up to 30 minutes.
for _ in range(360):
    if server.poll() is not None:
        raise RuntimeError("Server exited, see /content/fooocus-api.log")
    try:
        urllib.request.urlopen("http://127.0.0.1:8888/docs", timeout=3)
        print("Fooocus-API is up")
        break
    except Exception:
        time.sleep(5)
else:
    raise TimeoutError("Server not ready after 30 min, see /content/fooocus-api.log")

In [ ]:
import re, subprocess, time
!curl -sSL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tlog = open("/content/cloudflared.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8888"],
                          stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        url = m.group(0); break
    time.sleep(2)
print("FOOOCUS_API_URL=" + url if url else "No URL found, see /content/cloudflared.log")

In [ ]:
# Keep this cell running while you use the app; stopping it (or closing Colab) ends the session.
import time
while server.poll() is None and tunnel.poll() is None:
    time.sleep(30)
print("Server or tunnel stopped")